In [1]:
import requests
import pandas as pd
import json
import os
import boto3
import scrapy
from dotenv import load_dotenv


csv_url="https://tmopenlabbucket.s3.eu-west-3.amazonaws.com/City_Meteo_Rank.csv"


df = pd.read_csv(csv_url,index_col=0)

In [2]:
list_cities=df['city'].to_list()

In [3]:
import subprocess


# Liste des villes à envoyer à script.py
cities = list_cities

# Construire l'argument en ligne de commande
cmd = ['python3','booking_scrap_final.py','--cities'] + cities

# Exécuter le script en passant les villes comme argument
result = subprocess.run(cmd)


2026-07-28 08:23:26 [scrapy.utils.log] INFO: Scrapy 2.13.3 started (bot: scrapybot)
2026-07-28 08:23:26 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.0',
 'libxml2': '2.14.4',
 'cssselect': '1.3.0',
 'parsel': '1.10.0',
 'w3lib': '2.3.1',
 'Twisted': '25.5.0',
 'Python': '3.13.2 | packaged by Anaconda, Inc. | (main, Feb  6 2025, '
           '18:56:02) [GCC 11.2.0]',
 'pyOpenSSL': '25.1.0 (OpenSSL 3.0.16 11 Feb 2025)',
 'cryptography': '44.0.1',
 'Platform': 'Linux-6.11.0-29-generic-x86_64-with-glibc2.39'}
2026-07-28 08:23:26 [scrapy.addons] INFO: Enabled addons:
[]
2026-07-28 08:23:26 [scrapy.extensions.telnet] INFO: Telnet Password: a311c063f8fc8f13
2026-07-28 08:23:26 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats',
 'scrapy.extensions.throttle.AutoThrottle']
2026-07-28 08

In [4]:
df_booking=pd.read_json('hotels.json')
df_city_ccm=pd.read_csv('cities_lat_long_ccm.csv',index_col=0)


In [5]:
df_join=df_city_ccm.merge(df_booking, left_on='city', right_on='city', how='left')
df_join=df_join.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

In [6]:
df_join

,city,lat,lon,CCM,name,url,score,description,latitude,longitude
0,Le Havre,49.493898,0.107973,0.927,"The Originals Boutique, Hôtel d'Angleterre, Le...",https://www.booking.com/hotel/fr/comfort-d-ang...,7.6,This hotel is located in the town centre of Le...,49.494049,0.099366
1,Le Havre,49.493898,0.107973,0.927,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,https://www.booking.com/hotel/fr/la-parenthese...,9.1,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,49.496836,0.106893
2,Le Havre,49.493898,0.107973,0.927,Best Western ARThotel,https://www.booking.com/hotel/fr/art.en-gb.htm...,8.0,The Best Western Art Hotel is located in the h...,49.491194,0.106461
3,Le Havre,49.493898,0.107973,0.927,Aparthotel Adagio Access Le Havre Les Docks,https://www.booking.com/hotel/fr/adagio-access...,8.7,"Located in Le Havre, Aparthotel Adagio Access ...",49.487419,0.131511
4,Le Havre,49.493898,0.107973,0.927,Best Western Plus Le Havre Centre Gare,https://www.booking.com/hotel/fr/hotelterminus...,8.3,None,49.493344,0.124318
...,...,...,...,...,...,...,...,...,...,...
366,Grenoble,45.187560,5.735782,0.565,OKKO Hotels Grenoble Centre,https://www.booking.com/hotel/fr/okko-hotels-g...,8.4,None,45.184950,5.725990
367,Grenoble,45.187560,5.735782,0.565,"Le Grand Hôtel Grenoble, BW Premier Collection...",https://www.booking.com/hotel/fr/le-grand-gren...,8.6,In the heart of the historic city centre of th...,45.190815,5.728548
368,Grenoble,45.187560,5.735782,0.565,The Babel Community Hôtel - Grenoble Bastille,https://www.booking.com/hotel/fr/apt-paisible-...,8.6,"Situated conveniently in Grenoble, The Babel C...",45.195811,5.725447
369,Grenoble,45.187560,5.735782,0.565,RockyPop Grenoble Hotel,https://www.booking.com/hotel/fr/rockypop-gren...,8.5,"Situated in Grenoble, 1.6 km from Grenoble Tra...",45.186911,5.731293


In [7]:
df_join.to_csv('City_Meteo_Rank_Booking.csv')

In [8]:
### Upload to S3

load_dotenv()

aws_access_key_id =os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key =os.getenv('AWS_SECRET_ACCESS_KEY')
s3 = boto3.resource('s3',aws_access_key_id=aws_access_key_id,aws_secret_access_key=aws_secret_access_key)
bucket=s3.Bucket('tmopenlabbucket')

In [9]:
bucket.upload_file(Filename='City_Meteo_Rank_Booking.csv',Key='City_Meteo_Rank_Booking.csv', ExtraArgs={'ACL':'public-read'})